In [13]:
import asyncio
import httpx
import logging
import json
import sys
import os
from pathlib import Path
import time

In [15]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("CreditCardTest")


In [64]:
request_data = {
        "scenario_type": "credit_card",
        "parameters": {
            "url": "https://e.creditcard.ecitic.com/citiccard/ebank-ocp/ebankpc/myaccount.html"
        }
    }

In [70]:
async with httpx.AsyncClient() as client:
    logger.info("发送请求到API网关")

    response = await client.post(
        "http://localhost:8000/tasks",
        json=request_data,
        timeout=300  # 5分钟超时，因为登录可能需要时间
    )

2025-03-01 01:56:10,496 - CreditCardTest - INFO - 发送请求到API网关
2025-03-01 01:56:15,594 - httpx - INFO - HTTP Request: POST http://localhost:8000/tasks "HTTP/1.1 200 OK"


In [71]:
response.json()

{'status': 'error', 'message': ''}

In [75]:
request_data = {
            "url": "https://e.creditcard.ecitic.com/citiccard/ebank-ocp/ebankpc/myaccount.html"
}

In [76]:
async with httpx.AsyncClient() as client:
    logger.info("发送请求到API网关")

    response = await client.post(
        "http://localhost:8003/tools/browser/credit-card",
        json=request_data,
        timeout=300  # 5分钟超时，因为登录可能需要时间
    )

2025-03-01 02:07:22,346 - CreditCardTest - INFO - 发送请求到API网关
2025-03-01 02:08:33,480 - httpx - INFO - HTTP Request: POST http://localhost:8003/tools/browser/credit-card "HTTP/1.1 500 Internal Server Error"


In [74]:
response.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [19]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import socket
import time
import sys
import random
import threading
from datetime import datetime, timedelta


In [25]:
driver_path = '/usr/local/bin/chromedriver'
chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=chrome_options)

In [30]:
!$driver_path --version

ChromeDriver 133.0.6943.53 (9a80935019b0925b01cc21d254da203bc3986f04-refs/branch-heads/6943@{#1389})


In [26]:
# 获取页面上所有可能的选择器路径
def get_element_selectors(driver):
    # 使用JavaScript来获取所有元素的选择器
    js_script = """
    function getPathTo(element) {
        if (element.id !== '')
            return '#' + element.id;
        if (element === document.body)
            return 'body';

        var ix = 0;
        var siblings = element.parentNode.childNodes;
        for (var i = 0; i < siblings.length; i++) {
            var sibling = siblings[i];
            if (sibling === element)
                return getPathTo(element.parentNode) + ' > ' + element.tagName.toLowerCase() + ':nth-child(' + (ix + 1) + ')';
            if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
                ix++;
        }
    }
    
    var elements = document.querySelectorAll('*');
    var result = [];
    for (var i = 0; i < elements.length; i++) {
        var el = elements[i];
        if (el.textContent && el.textContent.trim() !== '' && el.style.display !== 'none') {
            result.push({
                text: el.textContent.trim().substring(0, 50),
                selector: getPathTo(el),
                classes: el.className,
                id: el.id
            });
        }
    }
    return result;
    """
    
    selectors = driver.execute_script(js_script)
    return selectors

selectors = get_element_selectors(driver)
print("页面上的元素选择器:")
for i, selector in enumerate(selectors):
    if i > 100:  # 限制输出数量
        print("...")
        break
    print(f"文本: {selector['text']}")
    print(f"选择器: {selector['selector']}")
    print(f"类名: {selector['classes']}")
    print(f"ID: {selector['id']}")
    print("---")

页面上的元素选择器:
文本: 首页-我的账户_中信银行信用卡会员中心
  
  




  
    
      
    

选择器: #undefined > html:nth-child(1)
类名: 
ID: 
---
文本: 首页-我的账户_中信银行信用卡会员中心
选择器: #undefined > html:nth-child(1) > head:nth-child(1)
类名: 
ID: 
---
文本: 首页-我的账户_中信银行信用卡会员中心
选择器: #undefined > html:nth-child(1) > head:nth-child(1) > title:nth-child(1)
类名: 
ID: 
---
文本: 欢迎您，王*典
      |
      
      |
      
    
  
  
 
选择器: body
类名: 
ID: 
---
文本: 欢迎您，王*典
      |
      
      |
      
    
  
  
 
选择器: body > div:nth-child(1)
类名: header
ID: 
---
文本: 欢迎您，王*典
      |
      
      |
选择器: body > div:nth-child(1) > div:nth-child(1)
类名: content
ID: 
---
文本: 欢迎您，王*典
      |
      
      |
选择器: body > div:nth-child(1) > div:nth-child(1) > div:nth-child(1)
类名: menu-area
ID: 
---
文本: 欢迎您，王*典
选择器: body > div:nth-child(1) > div:nth-child(1) > div:nth-child(1) > span:nth-child(1)
类名: 
ID: 
---
文本: 王*典
选择器: #userName
类名: 
ID: userName
---
文本: |
选择器: body > div:nth-child(1) > div:nth-child(1) > div:nth-child(1) > span:nth-child(2)
类名: line
ID:

In [27]:
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import re

def check_login_status(driver):
    """检查用户是否已登录"""
    # 方法1: 检查元素是否存在
    element_indicators = ["#userName", "#nameRare", ".ca_num", "#cardList"]
    for selector in element_indicators:
        try:
            element = driver.find_element(By.CSS_SELECTOR, selector)
            if element.is_displayed():
                print(f"元素检测: 找到登录指示器 {selector}")
                return True
        except:
            pass
    
    # 方法2: 检查文本内容
    js_script = """
    const textContent = document.body.innerText;
    return {
        hasName: textContent.includes('欢迎您'),
        hasBill: textContent.includes('本期应还金额'),
        hasDate: textContent.includes('到期还款日')
    };
    """
    
    result = driver.execute_script(js_script)
    if any(result.values()):
        print(f"文本检测: 找到登录指示 {result}")
        return True
    
    return False

def extract_account_info(driver):
    """提取账户信息"""
    # 使用JavaScript提取具体的文本值
    js_script = """
    const result = {
        welcomeMessage: null,
        billAmount: null,
        dueDate: null,
        cardNumber: null,
        minPayment: null
    };
    
    // 查找欢迎信息
    const welcomeElements = Array.from(document.querySelectorAll('*')).filter(el => 
        el.textContent && el.textContent.includes('欢迎您'));
    if (welcomeElements.length > 0) {
        result.welcomeMessage = welcomeElements[0].textContent.trim();
    }
    
    // 查找卡号
    const cardElements = document.querySelectorAll('.ca_num');
    if (cardElements.length > 0) {
        result.cardNumber = cardElements[0].textContent.trim();
    }
    
    // 查找账单金额
    const billElements = document.querySelectorAll('td:nth-child(1) > span.txt14');
    if (billElements.length > 0) {
        for (let el of billElements) {
            if (el.parentElement && el.parentElement.textContent.includes('本期应还金额')) {
                result.billAmount = el.textContent.trim();
                break;
            }
        }
    }
    
    // 查找最低还款金额
    const minPayElements = document.querySelectorAll('td:nth-child(2) > span.txt14');
    if (minPayElements.length > 0) {
        for (let el of minPayElements) {
            if (el.parentElement && el.parentElement.textContent.includes('最低还款金额')) {
                result.minPayment = el.textContent.trim();
                break;
            }
        }
    }
    
    // 查找到期还款日
    const dateElements = document.querySelectorAll('td:nth-child(2) > span.txt14');
    if (dateElements.length > 0) {
        for (let el of dateElements) {
            if (el.parentElement && el.parentElement.textContent.includes('到期还款日')) {
                result.dueDate = el.textContent.trim();
                break;
            }
        }
    }
    
    return result;
    """
    
    return driver.execute_script(js_script)

def extract_username_from_text(text):
    """从文本中提取用户名"""
    match = re.search(r'[欢迎您|您好]，([\w*]+)', text)
    return match.group(1) if match else "未知用户"

def display_formatted_info(data):
    """格式化显示账户信息"""
    # 提取用户名
    username = extract_username_from_text(data.get('welcomeMessage', ''))
    
    # 获取卡号
    card_number = data.get('cardNumber', '未获取到卡号')
    
    # 格式化显示
    print("\n" + "="*50)
    print("            中信银行信用卡账户信息摘要")
    print("="*50)
    
    print(f"\n👤 用户信息:")
    print(f"   用户名: {username}")
    print(f"   卡号: {card_number}")
    
    print(f"\n💰 账单信息:")
    print(f"   应还金额: ¥{data.get('billAmount', '未获取到')}")
    print(f"   最低还款: ¥{data.get('minPayment', '未获取到')}")
    print(f"   还款日期: {data.get('dueDate', '未获取到')}")
    
    print("\n" + "="*50)


In [28]:
def main(driver):
    """主函数: 检测登录状态并提取信息"""
    # 检查是否已登录
    is_logged_in = check_login_status(driver)
    
    if is_logged_in:
        print("用户已登录，正在提取账户信息...")
        # 提取账户信息
        account_info = extract_account_info(driver)
        # 格式化显示信息
        display_formatted_info(account_info)
        # 返回账户信息供进一步处理
        return account_info
    else:
        print("未检测到登录状态，请先登录账户。")
        return None

In [34]:
account_data = main(driver)

InvalidSessionIdException: Message: invalid session id
Stacktrace:
0   chromedriver                        0x000000010deb0bc8 chromedriver + 5766088
1   chromedriver                        0x000000010dea87ea chromedriver + 5732330
2   chromedriver                        0x000000010d9964c3 chromedriver + 414915
3   chromedriver                        0x000000010d9d8edf chromedriver + 687839
4   chromedriver                        0x000000010da0df56 chromedriver + 905046
5   chromedriver                        0x000000010da088b8 chromedriver + 882872
6   chromedriver                        0x000000010da07a47 chromedriver + 879175
7   chromedriver                        0x000000010d95f2bf chromedriver + 189119
8   chromedriver                        0x000000010de74100 chromedriver + 5517568
9   chromedriver                        0x000000010de78040 chromedriver + 5533760
10  chromedriver                        0x000000010de55c87 chromedriver + 5393543
11  chromedriver                     

In [60]:
def search_and_send_message(driver, contact_name, message):
    """
    Search for a contact and send them a message in WeChat Web
    
    Args:
        driver: Selenium WebDriver instance
        contact_name: Name of the contact to search for and message
        message: Text message to send
    """
    try:
        # Click on the search input field (magnifying glass icon)
        search_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".search_bar input"))
        )
        search_button.click()
        
        # Input the contact name in the search field
        search_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".search_bar input"))
        )
        search_input.clear()
        search_input.send_keys(contact_name)
        time.sleep(2)  # Wait for search results
        
        # Find and click on the contact in search results
        search_results = driver.find_elements(By.CSS_SELECTOR, ".contact_item")
        contact_found = False
        
        for result in search_results:
            try:
                name_element = result.find_element(By.CSS_SELECTOR, ".nickname")
                if contact_name in name_element.text:
                    result.click()
                    contact_found = True
                    print(f"Found and clicked on contact: {contact_name}")
                    break
            except:
                continue
        
        if not contact_found:
            print(f"Contact {contact_name} not found in search results")
            return False
        
        # Wait for chat window to load
        time.sleep(2)
        
        # Find the message input area and send message
        edit_area = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "editArea"))
        )
        
        # Clear any existing text and input our message
        edit_area.clear()
        edit_area.send_keys(message)
        
        # Find and click the send button
        send_button = driver.find_element(By.CSS_SELECTOR, ".btn_send")
        send_button.click()
        
        print(f"Message sent to {contact_name}: {message}")
        return True
        
    except Exception as e:
        print(f"Error searching for contact and sending message: {str(e)}")
        return False

In [62]:
# Example usage
contact_to_message = "朱天阳"  # The contact you want to search for
message_text = "我来试试：这是大模型发出的内容"

search_and_send_message(driver, contact_to_message, message_text)

Found and clicked on contact: 朱天阳
Message sent to 朱天阳: 我来试试：这是大模型发出的内容


True